In [ ]:
import os
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
import torch
import torch.nn as nn

In [ ]:
COMPETITION_DIR     = '/kaggle/input/competitions/birdclef-2026'
CHECKPOINT_PATH     = '/kaggle/input/datasets/joshgreen/birdclef-2026-checkpoint/checkpoint.pt'
PERCH_MODEL_PATH    = '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1'
TAXONOMY_PATH       = os.path.join(COMPETITION_DIR, 'taxonomy.csv')
TEST_SOUNDSCAPE_DIR = os.path.join(COMPETITION_DIR, 'test_soundscapes')

SAMPLE_RATE    = 32000
CHUNK_SECONDS  = 5
CHUNK_SAMPLES  = SAMPLE_RATE * CHUNK_SECONDS
EMBEDDING_DIM  = 1536
HIDDEN         = 512
PERCH_BATCH    = 256
MLP_BATCH      = 1024

taxonomy = pd.read_csv(TAXONOMY_PATH)
class_labels = sorted(taxonomy['primary_label'].astype(str).tolist())
n_classes = len(class_labels)
print(f'{n_classes} classes')

In [ ]:
tf.config.threading.set_intra_op_parallelism_threads(0)
tf.config.threading.set_inter_op_parallelism_threads(0)

perch = tf.saved_model.load(PERCH_MODEL_PATH)
infer = perch.signatures['serving_default']
_ = infer(inputs=tf.zeros([1, CHUNK_SAMPLES], dtype=tf.float32))  # warm up
print('Perch model loaded and warmed up')

In [ ]:
model = nn.Sequential(
    nn.Linear(EMBEDDING_DIM, HIDDEN),
    nn.ReLU(),
    nn.Dropout(0.0),
    nn.Linear(HIDDEN, n_classes),
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()
print(f'Checkpoint loaded (val_step={ckpt["val_step"]})')

In [ ]:
# Pass 1: collect all chunks and their row_ids
test_soundscapes = sorted(f for f in os.listdir(TEST_SOUNDSCAPE_DIR) if f.endswith('.ogg'))
print(f'{len(test_soundscapes)} test soundscapes')

all_chunks  = []
all_row_ids = []

for filename in test_soundscapes:
    sig, rate = sf.read(os.path.join(TEST_SOUNDSCAPE_DIR, filename), dtype='float32')
    if rate != SAMPLE_RATE:
        raise ValueError(f'{filename}: expected {SAMPLE_RATE}Hz, got {rate}Hz')
    soundscape_id = filename.split('.')[0]

    for i in range(0, len(sig), CHUNK_SAMPLES):
        chunk = sig[i : i + CHUNK_SAMPLES]
        if len(chunk) < CHUNK_SAMPLES:
            chunk = np.pad(chunk, (0, CHUNK_SAMPLES - len(chunk)))
        all_chunks.append(chunk)
        end_time = (i // CHUNK_SAMPLES + 1) * CHUNK_SECONDS
        all_row_ids.append(f'{soundscape_id}_{end_time}')

print(f'{len(all_chunks)} total chunks')

In [ ]:
# Pass 2: extract Perch embeddings in batches
n = len(all_chunks)
embeddings = np.zeros((n, EMBEDDING_DIM), dtype=np.float32)

for i in range(0, n, PERCH_BATCH):
    batch = np.stack(all_chunks[i : i + PERCH_BATCH])
    outputs = infer(inputs=tf.convert_to_tensor(batch))
    embeddings[i : i + len(batch)] = outputs['embedding'].numpy()

print(f'Embeddings: {embeddings.shape}')

In [ ]:
# Pass 3: MLP predictions in batches
all_scores = np.zeros((n, n_classes), dtype=np.float32)

with torch.no_grad():
    for i in range(0, n, MLP_BATCH):
        batch = torch.from_numpy(embeddings[i : i + MLP_BATCH]).to(device)
        scores = torch.sigmoid(model(batch)).cpu().numpy()
        all_scores[i : i + len(batch)] = scores

print('Predictions done')

In [ ]:
submission = pd.DataFrame(all_scores, columns=class_labels)
submission.insert(0, 'row_id', all_row_ids)
submission.to_csv('submission.csv', index=False)
print(f'submission.csv: {len(submission)} rows x {len(submission.columns)} columns')
submission.head()